<a id="interpolation-vs-fitting"></a>

$$
\text{插值与拟合 (Interpolation and Fitting): 已知一些离散的数据点 } (x_i, y_i) \text{, 求一个连续函数} \\ \quad \\
\text{插值 (Interpolation): 函数必须正好经过每一个数据点 } f(x_i) = y_i \\
\text{适用于数据准确 (例如查表, 实验测量很精确), 需要估计两点之间的值} \\ \quad \\
\text{拟合 (Fitting): 不要求经过每个点, 而是让整体误差最小 (例如最小二乘)} \\
\text{适用于数据有噪声, 想找背后的规律 (回归就是拟合)} \\ \quad \\
\text{1. 多项式插值的存在唯一性: n + 1 个点 (x 互不相同) 确定唯一的 n 次多项式 } p(x) = a_0 + a_1 x + \cdots + a_n x^n \\
\text{条件 } p(x_i) = y_i \text{ 写成线性方程组:} \\
\underbrace{\begin{bmatrix} 1 & x_0 & \cdots & x_0^n \\ 1 & x_1 & \cdots & x_1^n \\ \vdots & & & \vdots \\ 1 & x_n & \cdots & x_n^n \end{bmatrix}}_{V \text{ 范德蒙德矩阵 (Vandermonde)}} \begin{bmatrix} a_0 \\ a_1 \\ \vdots \\ a_n \end{bmatrix} = \begin{bmatrix} y_0 \\ y_1 \\ \vdots \\ y_n \end{bmatrix}, \quad \det V = \prod_{0 \leq i < j \leq n} (x_j - x_i) \neq 0 \\
\text{唯一性的另一个证明: 若两个 n 次多项式都插值, 它们的差是 n 次多项式却有 n + 1 个根, 只能恒为 0} \\
\text{问题: 范德蒙德矩阵的条件数随 n 增长极快, 直接解方程组数值上不稳定, 所以有下面的拉格朗日和牛顿形式}
$$

数学基础: [行列式](../math_basics/linear_algebra.ipynb#determinant) · [条件数](../math_basics/linear_algebra.ipynb#svd)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import lagrange as scipy_lagrange, CubicSpline

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

x_nodes = np.array([0.0, 1.0, 2.0, 4.0, 5.0])
y_nodes = np.array([1.0, 3.0, 2.0, 5.0, 4.0])

V = np.vander(x_nodes, increasing=True)
a = np.linalg.solve(V, y_nodes)
det_formula = np.prod([x_nodes[j] - x_nodes[i] for i in range(5) for j in range(i + 1, 5)])
print(f'det V = {np.linalg.det(V):.4f}, 公式 prod(x_j - x_i) = {det_formula:.4f}')
print(f'系数 (从低次到高次) = {np.round(a, 6)}, == np.polyfit(deg=4) ? {np.allclose(a[::-1], np.polyfit(x_nodes, y_nodes, 4))}')
print(f'经过所有点? {np.allclose(np.polyval(a[::-1], x_nodes), y_nodes)}')

for n in [5, 10, 15, 20]:
    xs = np.linspace(0, 1, n + 1)
    print(f'n = {n:>2}: 等距节点的范德蒙德矩阵条件数 = {np.linalg.cond(np.vander(xs, increasing=True)):.2e}')

<a id="lagrange-interpolation"></a>

$$
\text{2. 拉格朗日插值 (Lagrange Interpolation): 不解方程组, 直接构造} \\
\text{思路: 先为每个节点构造一个 "基函数" } l_i(x) \text{, 让它在 } x_i \text{ 处等于 1, 在其他节点处等于 0} \\
l_i(x_j) = \delta_{ij} = \begin{cases} 1, & i = j \\ 0, & i \neq j \end{cases} \\
\text{在其他 n 个节点处为 0 } \implies \text{分子取 } \prod_{j \neq i} (x - x_j) \text{ (n 次多项式, 这 n 个点都是根)} \\
\text{在 } x_i \text{ 处为 1 } \implies \text{除以它在 } x_i \text{ 处的值} \\
l_i(x) = \prod_{j \neq i} \frac{x - x_j}{x_i - x_j} \\
L(x) = \sum_{i=0}^{n} y_i \, l_i(x) \implies L(x_k) = \sum_i y_i \delta_{ik} = y_k \quad (\text{自动经过所有点}) \\ \quad \\
\text{插值误差 (和泰勒余项形式相似): } f(x) - L(x) = \frac{f^{(n+1)}(\xi)}{(n+1)!} \prod_{i=0}^{n} (x - x_i) \\
\text{(证明: 固定 x, 构造 } g(t) = f(t) - L(t) - c \prod (t - x_i) \text{, 选 c 让 g(x) = 0, g 有 n + 2 个零点,} \\
\text{由罗尔定理 } g^{(n+1)} \text{ 至少有一个零点 } \xi \text{, 而 } g^{(n+1)}(\xi) = f^{(n+1)}(\xi) - c (n+1)! = 0 \text{)}
$$

数学基础: [泰勒展开与余项](../math_basics/calculus.ipynb#taylor)

In [ ]:
def lagrange_basis(x_nodes, i, x):
    terms = [(x - x_nodes[j]) / (x_nodes[i] - x_nodes[j]) for j in range(len(x_nodes)) if j != i]
    return np.prod(terms, axis=0)

def lagrange_interp(x_nodes, y_nodes, x):
    return sum(y_nodes[i] * lagrange_basis(x_nodes, i, x) for i in range(len(x_nodes)))

x_plot = np.linspace(-0.2, 5.2, 400)
L_manual = lagrange_interp(x_nodes, y_nodes, x_plot)
print(f'手写拉格朗日 == scipy lagrange ? {np.allclose(L_manual, scipy_lagrange(x_nodes, y_nodes)(x_plot))}')
print(f'== 范德蒙德解出的多项式 ? {np.allclose(L_manual, np.polyval(a[::-1], x_plot))}')
print(f'基函数在节点处的值 (单位矩阵) = \n{np.round(np.array([lagrange_basis(x_nodes, i, x_nodes) for i in range(5)]), 10) + 0.0}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for i in range(5):
    axs[0].plot(x_plot, lagrange_basis(x_nodes, i, x_plot), label=f'l_{i}(x)')
axs[0].plot(x_nodes, np.zeros(5), 'k.')
axs[0].set_title('Lagrange basis')
axs[1].plot(x_plot, L_manual, label='L(x)')
axs[1].plot(x_nodes, y_nodes, 'ko', label='data')
axs[1].set_title('interpolating polynomial')
plot_nicely()
plt.show()

# 误差公式验证: f = sin, 4 次插值, 误差 <= max|f^(5)| / 5! * |prod(x - x_i)|, |f^(5)| = |cos| <= 1
xn = np.linspace(0, np.pi, 5)
xx = np.linspace(0, np.pi, 1000)
err = np.abs(np.sin(xx) - lagrange_interp(xn, np.sin(xn), xx))
bound = np.abs(np.prod([xx - xi for xi in xn], axis=0)) / 120
print(f'sin x 4 次插值: 最大误差 = {err.max():.2e}, 误差界 = {bound.max():.2e}, 处处满足误差界? {np.all(err <= bound + 1e-15)}')

<a id="newton-interpolation"></a>

$$
\text{3. 牛顿插值 (Newton Interpolation): 增加一个节点时, 不用全部重新计算} \\
N(x) = c_0 + c_1 (x - x_0) + c_2 (x - x_0)(x - x_1) + \cdots + c_n (x - x_0)\cdots(x - x_{n-1}) \\
\text{逐个代入节点 (每代入一个, 后面的项都为 0, 像解下三角方程组):} \\
x = x_0: \ c_0 = y_0 \\
x = x_1: \ y_1 = c_0 + c_1(x_1 - x_0) \implies c_1 = \frac{y_1 - y_0}{x_1 - x_0} \\
\text{一般地 } c_k = f[x_0, x_1, \cdots, x_k] \text{ (差商 Divided Difference), 递推定义:} \\
f[x_i] = y_i, \quad f[x_i, \cdots, x_{i+k}] = \frac{f[x_{i+1}, \cdots, x_{i+k}] - f[x_i, \cdots, x_{i+k-1}]}{x_{i+k} - x_i} \\
\text{一阶差商就是割线斜率 (导数的离散版本), k 阶差商} \approx \frac{f^{(k)}}{k!} \\
\text{增加节点 } x_{n+1} \text{ 时, 只需在后面加一项 } c_{n+1}(x - x_0)\cdots(x - x_n) \text{, 前面的系数不变} \\
\text{由唯一性, 牛顿形式和拉格朗日形式是同一个多项式, 只是写法不同}
$$

数学基础: [导数定义 (割线斜率)](../math_basics/calculus.ipynb#derivative)

In [ ]:
def divided_differences(x_nodes, y_nodes):
    n = len(x_nodes)
    table = np.zeros((n, n))
    table[:, 0] = y_nodes
    for k in range(1, n):
        for i in range(n - k):
            table[i, k] = (table[i + 1, k - 1] - table[i, k - 1]) / (x_nodes[i + k] - x_nodes[i])
    return table          # 第 0 行就是牛顿系数 c_0, c_1, ...

def newton_eval(x_nodes, coef, x):
    result = np.zeros_like(x, dtype=float) + coef[-1]
    for k in range(len(coef) - 2, -1, -1):       # 类似秦九韶 / Horner 算法
        result = result * (x - x_nodes[k]) + coef[k]
    return result

table = divided_differences(x_nodes, y_nodes)
coef = table[0]
print(f'差商表 = \n{np.round(table, 4)}')
print(f'牛顿插值 == 拉格朗日插值 ? {np.allclose(newton_eval(x_nodes, coef, x_plot), L_manual)}')

# 增加一个节点: 前面的系数不变
x_more, y_more = np.append(x_nodes, 3.0), np.append(y_nodes, 2.5)
coef_more = divided_differences(x_more, y_more)[0]
print(f'加节点前系数 = {np.round(coef, 6)}')
print(f'加节点后系数 = {np.round(coef_more, 6)} (前 5 个不变)')

# k 阶差商 ≈ f^(k) / k!
xs = np.array([1.0, 1.01, 1.02, 1.03])
print(f'exp 的 3 阶差商 = {divided_differences(xs, np.exp(xs))[0, 3]:.6f}, e^xi / 3! (xi 取区间中点 1.015) = {np.exp(1.015) / 6:.6f}')

<a id="runge-phenomenon"></a>

$$
\text{4. 龙格现象 (Runge Phenomenon): 等距节点的高次插值, 在区间两端会剧烈振荡} \\
\text{例: } f(x) = \frac{1}{1 + 25x^2}, \ x \in [-1, 1] \text{, 节点越多, 端点附近误差反而越大} \\
\text{原因: 误差公式中 } \prod (x - x_i) \text{ 在等距节点下, 端点附近的值比中间大得多, 而且 } f^{(n+1)} \text{ 增长很快} \\ \quad \\
\text{解决方法:} \\
\text{(1) 切比雪夫节点 (Chebyshev Nodes): } x_k = \cos\frac{(2k + 1)\pi}{2(n + 1)}, \ k = 0, \cdots, n \text{ (两端更密)} \\
\quad \text{它们让 } \max_{x \in [-1, 1]} \left|\prod (x - x_k)\right| \text{ 达到最小值 } 2^{-n} \\
\text{(2) 分段低次插值: 分段线性, 三次样条 (下面)}
$$

In [ ]:
runge = lambda x: 1 / (1 + 25 * x**2)
xx = np.linspace(-1, 1, 2001)

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(xx, runge(xx), 'k', linewidth=2, label='f(x)')
axs[1].plot(xx, runge(xx), 'k', linewidth=2, label='f(x)')
for n in [6, 10, 14]:
    x_eq = np.linspace(-1, 1, n + 1)
    x_ch = np.cos((2 * np.arange(n + 1) + 1) * np.pi / (2 * (n + 1)))
    p_eq = lagrange_interp(x_eq, runge(x_eq), xx)
    p_ch = lagrange_interp(x_ch, runge(x_ch), xx)
    axs[0].plot(xx, p_eq, label=f'equispaced n={n}')
    axs[1].plot(xx, p_ch, label=f'Chebyshev n={n}')
    prod_eq = np.abs(np.prod([xx - xi for xi in x_eq], axis=0)).max()
    prod_ch = np.abs(np.prod([xx - xi for xi in x_ch], axis=0)).max()
    print(f'n = {n:>2}: 最大误差 等距 = {np.abs(p_eq - runge(xx)).max():.4f}, 切比雪夫 = {np.abs(p_ch - runge(xx)).max():.4f}; '
          f'max|prod| 等距 = {prod_eq:.2e}, 切比雪夫 = {prod_ch:.2e} (理论 2^-n = {2.0**-n:.2e})')
axs[0].set_ylim(-1, 2)
axs[1].set_ylim(-1, 2)
plot_nicely()
plt.show()

<a id="cubic-spline"></a>

$$
\text{5. 三次样条插值 (Cubic Spline): 每个小区间用一个三次多项式, 并且接点处光滑} \\
\text{n + 1 个节点, n 个区间, 第 i 个区间 } [x_i, x_{i+1}] \text{ 上的三次多项式 } S_i(x) \text{, 共 4n 个未知系数} \\
\text{条件: 插值 } S_i(x_i) = y_i, S_i(x_{i+1}) = y_{i+1} \text{ (2n 个), 一阶导连续 (n - 1 个), 二阶导连续 (n - 1 个)} \\
\text{共 4n - 2 个方程, 还差 2 个边界条件: 自然样条 (Natural Spline) } S''(x_0) = S''(x_n) = 0 \\ \quad \\
\text{推导 (用节点处的二阶导 } M_i = S''(x_i) \text{ 作为未知数):} \\
S_i \text{ 是三次多项式, } S_i'' \text{ 是一次函数, 由两端的值线性插值, 记 } h_i = x_{i+1} - x_i \text{:} \\
S_i''(x) = M_i \frac{x_{i+1} - x}{h_i} + M_{i+1} \frac{x - x_i}{h_i} \quad (\text{这样二阶导自动连续}) \\
\text{积分两次, 两个积分常数由 } S_i(x_i) = y_i, S_i(x_{i+1}) = y_{i+1} \text{ 确定:} \\
S_i(x) = \frac{M_i (x_{i+1} - x)^3 + M_{i+1} (x - x_i)^3}{6 h_i} + \left(\frac{y_i}{h_i} - \frac{M_i h_i}{6}\right)(x_{i+1} - x) + \left(\frac{y_{i+1}}{h_i} - \frac{M_{i+1} h_i}{6}\right)(x - x_i) \\
\text{(验证: x = } x_i \text{ 时只剩 } \frac{M_i h_i^3}{6 h_i} + y_i - \frac{M_i h_i^2}{6} = y_i \text{)} \\
\text{求导, 在 } x_i \text{ 处的右导数 (第 i 段) 和左导数 (第 i - 1 段):} \\
S_i'(x_i) = \frac{y_{i+1} - y_i}{h_i} - \frac{h_i}{6}(2M_i + M_{i+1}), \quad S_{i-1}'(x_i) = \frac{y_i - y_{i-1}}{h_{i-1}} + \frac{h_{i-1}}{6}(M_{i-1} + 2M_i) \\
\text{令两者相等 (一阶导连续), 整理得 } i = 1, \cdots, n - 1 \text{ 的三对角方程组 (Tridiagonal System):} \\
h_{i-1} M_{i-1} + 2(h_{i-1} + h_i) M_i + h_i M_{i+1} = 6\left(\frac{y_{i+1} - y_i}{h_i} - \frac{y_i - y_{i-1}}{h_{i-1}}\right), \quad M_0 = M_n = 0 \\
\text{对角元 } 2(h_{i-1} + h_i) > h_{i-1} + h_i \text{ (严格对角占优), 所以矩阵可逆, 解唯一} \\
\text{三对角方程组可以用追赶法 (Thomas Algorithm) 在 O(n) 内求解 (高斯消元只需消去一条次对角线)} \\ \quad \\
\text{性质: 在所有经过这些点, 二阶导连续的函数中, 自然样条的 } \int (S'')^2 dx \text{ 最小 (最 "不弯曲", 这也是 "样条" 名字的来源: 绘图用的弹性木条)}
$$

数学基础: [积分](../math_basics/calculus.ipynb#integral) · [矩阵的逆](../math_basics/linear_algebra.ipynb#inverse)

In [ ]:
def thomas(lower, diag, upper, rhs):
    # 追赶法: lower[i] 是第 i 行的次对角元 (lower[0] 不用), upper[i] 是第 i 行的超对角元
    n = len(diag)
    c, d = np.zeros(n), np.zeros(n)
    c[0], d[0] = upper[0] / diag[0], rhs[0] / diag[0]
    for i in range(1, n):
        denom = diag[i] - lower[i] * c[i - 1]
        c[i] = upper[i] / denom if i < n - 1 else 0.0
        d[i] = (rhs[i] - lower[i] * d[i - 1]) / denom
    x = np.zeros(n)
    x[-1] = d[-1]
    for i in range(n - 2, -1, -1):
        x[i] = d[i] - c[i] * x[i + 1]
    return x

def natural_spline(x_nodes, y_nodes):
    n = len(x_nodes) - 1
    h = np.diff(x_nodes)
    lower = np.r_[0, h[1:n - 1]]
    diag = 2 * (h[:-1] + h[1:])
    upper = np.r_[h[1:n - 1], 0]
    rhs = 6 * ((y_nodes[2:] - y_nodes[1:-1]) / h[1:] - (y_nodes[1:-1] - y_nodes[:-2]) / h[:-1])
    M = np.zeros(n + 1)
    M[1:-1] = thomas(lower, diag, upper, rhs)

    def S(x):
        x = np.asarray(x, dtype=float)
        i = np.clip(np.searchsorted(x_nodes, x) - 1, 0, n - 1)
        xi, xi1, hi = x_nodes[i], x_nodes[i + 1], h[i]
        return ((M[i] * (xi1 - x)**3 + M[i + 1] * (x - xi)**3) / (6 * hi)
                + (y_nodes[i] / hi - M[i] * hi / 6) * (xi1 - x)
                + (y_nodes[i + 1] / hi - M[i + 1] * hi / 6) * (x - xi))
    return S, M, (lower, diag, upper, rhs)

S, M, (lower, diag, upper, rhs) = natural_spline(x_nodes, y_nodes)
A_full = np.diag(diag) + np.diag(lower[1:], -1) + np.diag(upper[:-1], 1)
print(f'三对角矩阵 = \n{A_full}')
print(f'追赶法 == np.linalg.solve ? {np.allclose(M[1:-1], np.linalg.solve(A_full, rhs))}')

cs = CubicSpline(x_nodes, y_nodes, bc_type='natural')
x_in = np.linspace(0, 5, 500)
print(f'手写自然样条 == scipy CubicSpline(natural) ? {np.allclose(S(x_in), cs(x_in))}')
print(f'节点处二阶导 M = {np.round(M, 6)}, scipy = {np.round(cs(x_nodes, 2), 6) + 0.0}')

# 最小弯曲: 比较自然样条和其他经过同样点的光滑函数 (not-a-knot 样条, 4 次插值多项式) 的 ∫ (S'')^2
from scipy.integrate import quad
bend = lambda f2: quad(lambda x: f2(x)**2, 0, 5, limit=200)[0]
cs_nak = CubicSpline(x_nodes, y_nodes, bc_type='not-a-knot')
poly2 = np.polyder(np.polyfit(x_nodes, y_nodes, 4), 2)
print(f'∫(S\'\')^2: 自然样条 = {bend(lambda x: cs(x, 2)):.4f}, not-a-knot 样条 = {bend(lambda x: cs_nak(x, 2)):.4f}, 4 次多项式 = {bend(lambda x: np.polyval(poly2, x)):.4f}')

plt.figure(figsize=(4, 3))
plt.plot(x_nodes, y_nodes, 'ko', label='data')
plt.plot(x_in, np.interp(x_in, x_nodes, y_nodes), ':', label='piecewise linear')
plt.plot(x_in, S(x_in), label='natural cubic spline')
plt.plot(x_in, lagrange_interp(x_nodes, y_nodes, x_in), '--', label='degree-4 polynomial')
plot_nicely()
plt.show()

# Runge 函数: 样条随节点增多一致收敛
for n in [6, 10, 20, 40]:
    xe = np.linspace(-1, 1, n + 1)
    Sr, _, _ = natural_spline(xe, runge(xe))
    print(f'Runge 函数 n = {n:>2}: 样条最大误差 = {np.abs(Sr(xx) - runge(xx)).max():.2e}, 分段线性 = {np.abs(np.interp(xx, xe, runge(xe)) - runge(xx)).max():.2e}')

<a id="least-squares-fitting"></a>

$$
\text{6. 最小二乘拟合 (Least Squares Fitting): 数据有噪声时, 不要求经过每个点} \\
\text{用 m 次多项式 (m 远小于数据点数 n) 拟合: } \min_\beta \sum_{i=1}^{n} \left(y_i - \sum_{k=0}^{m} \beta_k x_i^k\right)^2 = \|y - X\beta\|^2 \\
X \text{ 是 n 行 m + 1 列的范德蒙德矩阵 (不再是方阵)} \\
\text{对 } \beta \text{ 求梯度令为 0 (矩阵求导): } \nabla_\beta \|y - X\beta\|^2 = -2X^T(y - X\beta) = 0 \\
X^T X \beta = X^T y \quad (\text{正规方程 Normal Equation}) \\
\text{几何意义: } X\hat{\beta} \text{ 是 y 在 X 的列空间上的投影, 残差和列空间正交} \\ \quad \\
\text{插值 = 拟合的特殊情况 (m = n - 1 时残差为 0), 但会把噪声也拟合进去 (过拟合 Overfitting)} \\
\text{选择次数: 在新数据 (测试集) 上的误差最小, 而不是训练数据上的误差最小}
$$

数学基础: [矩阵求导](../math_basics/linear_algebra.ipynb#matrix-calculus) · [投影矩阵与最小二乘](../math_basics/linear_algebra.ipynb#projection) · 相关模型: [多元线性回归](linear_regression_model/multivariate_linear_regression.ipynb) · [非线性回归 (多项式回归)](linear_regression_model/non_linear_regression.ipynb)

In [ ]:
rng = np.random.default_rng(0)
true_f = lambda x: np.sin(2 * np.pi * x)
x_train = np.sort(rng.uniform(0, 1, 15))
y_train = true_f(x_train) + rng.normal(0, 0.2, 15)
x_test = rng.uniform(0, 1, 200)
y_test = true_f(x_test) + rng.normal(0, 0.2, 200)

def ls_polyfit(x, y, m):
    X = np.vander(x, m + 1, increasing=True)
    return np.linalg.solve(X.T @ X, X.T @ y)

beta3 = ls_polyfit(x_train, y_train, 3)
X3 = np.vander(x_train, 4, increasing=True)
residual = y_train - X3 @ beta3
print(f'正规方程 == np.polyfit ? {np.allclose(beta3[::-1], np.polyfit(x_train, y_train, 3))}')
print(f'残差和列空间正交 X^T r = {np.round(X3.T @ residual, 10) + 0.0}')

degrees = range(0, 13)
train_err, test_err = [], []
for m in degrees:
    coef_m = np.polyfit(x_train, y_train, m)
    train_err.append(np.mean((np.polyval(coef_m, x_train) - y_train)**2))
    test_err.append(np.mean((np.polyval(coef_m, x_test) - y_test)**2))
best_m = list(degrees)[int(np.argmin(test_err))]
print(f'测试误差最小的次数 = {best_m}; 14 次 (插值) 的训练误差 = {np.mean((lagrange_interp(x_train, y_train, x_train) - y_train)**2):.2e}, '
      f'测试误差 = {np.mean((lagrange_interp(x_train, y_train, x_test) - y_test)**2):.2e}')

xf = np.linspace(0, 1, 400)
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(x_train, y_train, 'ko', label='noisy data')
axs[0].plot(xf, true_f(xf), 'k--', label='true sin(2 pi x)')
axs[0].plot(xf, np.polyval(np.polyfit(x_train, y_train, best_m), xf), label=f'LS degree {best_m}')
axs[0].plot(xf, lagrange_interp(x_train, y_train, xf), ':', label='interpolation (degree 14)')
axs[0].set_ylim(-2, 2)
axs[1].semilogy(list(degrees), train_err, 'o-', label='train MSE')
axs[1].semilogy(list(degrees), test_err, 's-', label='test MSE')
axs[1].set_xlabel('degree')
plot_nicely()
plt.show()